# 08 — Reading Documents

## What you'll learn
- How to extract text from a **PDF** with `read_pdf` and why a text layer matters.
- How to load a **spreadsheet** with `read_excel` and what "first sheet by default" means.
- How to pull text out of a **Word `.docx`** with `read_docx`.
- Where real financial research actually comes from (10-K/10-Q filings, earnings slides, broker exports).
- How to feed extracted prose into sentiment analysis — and why analyst prose is *not* a headline.

Everything here uses the three small sample files in `sample_docs/`, so this notebook needs no network.
They stand in for the real documents you will meet: an earnings summary (PDF), a quarterly
spreadsheet (Excel) and an analyst note (Word).

In [1]:
%matplotlib inline

from pathlib import Path

import pandas as pd
import seaborn as sns

from stocklearn.docs import read_pdf, read_excel, read_docx
from stocklearn.sentiment import analyze_text

# Jupyter starts the kernel inside notebooks/, while sample_docs/ lives in the repo root
# next to pyproject.toml. Walk up until we find it so these paths work from anywhere,
# instead of depending on the notebook being launched from one particular folder.
ROOT = Path.cwd()
while not (ROOT / "sample_docs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DOCS = ROOT / "sample_docs"
print("repo root:", ROOT)
print("documents:", sorted(p.name for p in DOCS.glob("*")))

# We keep the course convention of one top variable for the symbol you are studying.
# This notebook reads local files rather than market data, so `ticker` is unused here —
# swap it when you adapt these document paths for your own research.
ticker = "AAPL"

sns.set_theme(style="whitegrid")
print("pandas:", pd.__version__)
print("document readers ready: read_pdf, read_excel, read_docx")

repo root: /Users/piyush/Documents/stock-market-learning
documents: ['earnings_note.pdf', 'quarterly_financials.xlsx', 'research_note.docx']
pandas: 3.0.6
document readers ready: read_pdf, read_excel, read_docx


## Why documents?

Prices arrive as clean numbers, but a huge amount of financial information only exists as **prose or
tables inside files**:

- **PDF** — annual reports (10-K), quarterly reports (10-Q), earnings press releases, slide decks.
- **Excel** — broker/fundamental data exports, your own models, finance-team workbooks.
- **Word** — analyst research notes, internal memos, investment-committee write-ups.

To analyze any of it with Python you must first **extract** the text or the table into a `str` or a
`DataFrame`. Extraction quality decides everything downstream — bad text in, bad conclusions out.

Let's start with the PDF.

In [2]:
pdf_text = read_pdf(DOCS / "earnings_note.pdf")

print(pdf_text)
print()
print("extracted characters:", len(pdf_text))

SampleCo Q3 2025 Earnings Summary
SampleCo reported third-quarter revenue of $1.2 billion, up 15% year over year, driven by strong
demand in its core subscription business.
Net income was $240 million, or $1.20 per diluted share. Management reiterated full-year guidance and
highlighted margin expansion as input costs eased.
Metric
Q3 2025
Q2 2025
Revenue
$1.20B
$1.04B
Net income
$240M
$205M
EPS
$1.20
$1.02


extracted characters: 410


### What `read_pdf` just did

`read_pdf` opens the file with **pypdf**, walks every page, calls `page.extract_text()` and joins the
pages with blank lines. It returns one plain string.

**The crucial caveat:** a PDF normally stores *glyphs plus a text layer*, and `extract_text()` reads that
layer. A **scanned** or image-only PDF (a photo of a page, a fax, many older filings) has *no text
layer* — `read_pdf` returns an empty string, not an error. That is what **OCR** (optical character
recognition) is for: it converts the pixels back into text. pypdf does not do OCR; you would add a tool
such as `pytesseract` for that.

So always check the length: `len(read_pdf(path)) == 0` is your signal that OCR is needed.

## Spreadsheets: `read_excel`

Excel files can hold many **sheets** and often begin with title or blank **header rows**. `read_excel`
returns a `DataFrame` from the **first sheet** by default.

In [3]:
financials = read_excel(DOCS / "quarterly_financials.xlsx")

print("shape:", financials.shape)
financials

shape: (3, 4)


,Line item ($M),Q1 2025,Q2 2025,Q3 2025
0,Revenue,900,1040,1200
1,Gross Profit,540,640,750
2,Net Income,170,205,240


### The knobs that matter on real files

- **`sheet_name=`** — pick a sheet by name or index, e.g. `sheet_name="Income Statement"` or
  `sheet_name=1`. The default is always the *first* sheet, which is not always the one you want.
- **`header=`** — real broker exports frequently have a logo/title/date block above the column names.
  `pd.read_excel(path, header=3)` says "the header is on the 4th row"; `header=None` treats every row as data.
- **`skiprows=` / `usecols=`** — drop banner rows or select a subset of columns.
- **dtypes and blanks** — numbers may arrive as text (`"1,200"`), merged cells become `NaN`, and totals
  rows can sneak in as data. Inspect `.dtypes` before trusting a column of numbers.

Here we were lucky: row 0 already held the column names (`Line item ($M)`, `Q1 2025`, …).

## Word documents: `read_docx`

A `.docx` is a zip of XML parts. `read_docx` uses **python-docx**, iterates the document's
**paragraphs** and joins the non-empty ones with newlines.

In [4]:
note_text = read_docx(DOCS / "research_note.docx")

print(note_text)
print()
print("extracted characters:", len(note_text))

SampleCo Equity Research Note
SampleCo delivered a clean quarter with revenue of $1.2B (+15% YoY) and net income of $240M. Gross margin held above 62%, consistent with the prior three quarters.
Risks: customer concentration in the top ten accounts and a strong dollar could pressure reported growth. We remain constructive on the subscription transition and see margin upside if input costs stay flat.

extracted characters: 401


### Paragraphs only — tables are separate

`read_docx` sees **paragraphs**: headings, body text, bullets. It does **not** see **tables**, which
live in a separate collection. To read a table you would open the file directly:

```python
import docx
document = docx.Document("sample_docs/research_note.docx")
for table in document.tables:          # each table has .rows and .cells
    for row in table.rows:
        print([cell.text for cell in row.cells])
```

That same split applies to PDFs (text layer vs. layout) and Excel (values vs. formatting): the reader
gives you *content*, not the visual page.

## Tie-in: sentiment on extracted prose

Once text is a string, the rest of the toolkit applies. We can sentence-split the research note and run
our VADER wrapper `analyze_text` on the opening lines — the same function notebook 07 uses on headlines.

In [5]:
# Take the first two sentences of the analyst note.
sentences = note_text.split(". ")
opening = ". ".join(sentences[:2]).strip()
if not opening.endswith("."):
    opening += "."

print("text analysed:")
print(" ", opening.replace("\n", " "))
print()
print("scores:", analyze_text(opening))
print()
compound = analyze_text(opening)["compound"]
label = "positive" if compound >= 0.05 else "negative" if compound <= -0.05 else "neutral"
print(f"compound {compound:+.3f} -> {label}")

text analysed:
  SampleCo Equity Research Note SampleCo delivered a clean quarter with revenue of $1.2B (+15% YoY) and net income of $240M. Gross margin held above 62%, consistent with the prior three quarters. Risks: customer concentration in the top ten accounts and a strong dollar could pressure reported growth.

scores: {'neg': 0.128, 'neu': 0.692, 'pos': 0.18, 'compound': 0.4588}

compound +0.459 -> positive


### Analyst prose is not a headline

`analyze_text` returns VADER's four scores (`neg`, `neu`, `pos`, `compound`). The reading above is
*mildly* positive — but treat it with suspicion here:

- **Headlines are written to be punchy.** Analyst prose is hedged, comparative and full of caveats
  ("risks", "could pressure"). VADER was tuned on social-media/headline text, so it under- or
  over-reads formal writing.
- **Sentiment is not direction.** A note can be *positive in tone* while recommending caution, or a
  company can beat estimates while the stock falls on guidance. The score measures word polarity, not
  expected return.
- **Long documents dilute.** Averaging sentiment over a whole 10-K washes out into ~neutral; per-section
  or per-sentence scoring is more informative than one number for the entire file.

Use it as one weak signal, never a verdict.

## Real-world sources and pitfalls

Where this data actually comes from, and what goes wrong:

| Source | Format | Notes |
|---|---|---|
| **SEC EDGAR** (10-K/10-Q/8-K) | PDF/HTML | Free and authoritative. HTML filings parse far more reliably than PDFs. |
| **Earnings press releases & slides** | PDF/PPT | Often image-heavy; check for an empty extraction before trusting it. |
| **Broker / data-vendor exports** | Excel/CSV | Header rows, merged cells, `"1,234"` as text, footnotes-as-rows. |
| **Analyst research notes** | Word/PDF | Legally restricted to redistribute; good for sentiment experiments locally. |
| **Your own models** | Excel | Named sheets and formulas — `pandas` sees *cached values*, not formulas. |

Three habits that save you:

1. **Always print `len(text)` or `df.shape` first.** Empty extraction is silent failure.
2. **Inspect, don't assume,** the table shape: `.head()`, `.dtypes`, `.columns` — Yahoo-vendor drift applies
   to spreadsheets too.
3. **Extraction quality decides everything downstream.** Sentiment, keyword search and LLM summarization
   are all only as good as the text you hand them; garbage in, confident garbage out.

For regulated filings, prefer the **HTML** version on EDGAR — it keeps tables and text structure that a
PDF's text layer flattens away.

## Recap / try this

You can now turn the three document formats that carry real research into Python objects, and you know
the failure mode (empty text = scanned PDF, needs OCR) and the table trap (Word tables live outside
`paragraphs`).

Experiments to try:

1. Print `read_excel("sample_docs/quarterly_financials.xlsx").dtypes` — why is `Line item ($M)` `object`
   while the quarter columns are integers? What would you do if a number came through as `"1,200"`?
2. Score the *second* paragraph (`sentences[2:]`, the "Risks:" part) and compare its compound score with
   the opening. Does the tone flip to negative?
3. Download a real 10-Q PDF from SEC EDGAR, run `read_pdf` on it and print `len(...)`. If it is near
   zero, the file is image-based — you now know you need OCR.
4. Re-read `sample_docs/research_note.docx` with `python-docx` and print `len(doc.tables)` — confirm
   that `read_docx` (paragraphs only) would have missed those tables.

Next: **09_full_analysis** combines prices, indicators, fundamentals, news sentiment and this document
skill into one descriptive "will it go up?" report.